In [25]:
import pandas as pd
from datasets import load_dataset
from datetime import datetime
from groq import Groq
import os
import json


In [ ]:
os.environ["GROQ_API_KEY"] = "API-KEY"
client = Groq()

In [34]:
dataset = load_dataset("gsm8k", "main")


test_data = dataset["test"].select(range(10))



test_subset = dataset["test"].select(range(5))
for sample in test_subset:
    print("\nquestion:",sample["question"])
    print("\nanswer:",sample["answer"])
    print("-"*30)


question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

answer: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
------------------------------

question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

answer: It takes 2/2=<<2/2=1>>1 bolt of white fiber
So the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric
#### 3
------------------------------

question: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

answer: The cost of the house and repairs came out to 80,000+50,000=$<<8

In [ ]:
current_model="openai/gpt-oss-120b"

def query_groq(prompt, model=current_model):
     
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    
    return response.choices[0].message.content

In [ ]:
def build_prompt(question, student_solution):
    return f"""

#system propmt - general instructions and grading guidelines (remains the same for all exercises)



You are a math grader.

Grade the student's solution using the following rubric:

- Correct final answer (5 points)
- Correct reasoning steps (3 points)
- Clarity and explanation (2 points)

Total possible: 10 points.

Return your response in EXACT JSON format:
{{
  "score": <integer 0-10>,
  "feedback": "<short explanation>"
}}



#user prompt - exercise specific info : the question, the predefined correct answer, student submission,
                                        grading rubric, TA graded examples


Question:
{question}

Student solution:
{student_solution}
"""

In [ ]:

results = []

for i, sample in enumerate(test_data):
    print(f"Processing sample {i+1}")

    question = sample["question"]

    # GSM8K answer as "student solution" 
    student_solution = sample["answer"]

    prompt = build_prompt(question, student_solution)

    raw_output = query_groq(prompt, model=current_model)

    # parse JSON returned by the LLM
    try:
        parsed = json.loads(raw_output)
        score = parsed["score"]
        feedback = parsed["feedback"]
    except:
        score = None
        feedback = "JSON parsing failed"

    results.append({
        "question_id": i,
        "model": current_model,
        "score": score,
        "feedback": feedback,
        "timestamp": datetime.now()
    })

Processing sample 1
Processing sample 2
Processing sample 3
Processing sample 4
Processing sample 5
Processing sample 6
Processing sample 7
Processing sample 8
Processing sample 9
Processing sample 10


In [31]:
results_df = pd.DataFrame(results)
new_model_name = current_model.replace("/", "_")
results_df.to_csv(f"../results/rubric_gsm8k_{new_model_name}.csv", index=False)